(2:sampling_nb)=
# Notebook: sampling

If not automatically done, click {fa}`rocket` --> {guilabel}`Live Code` on the top right corner of this screen and then wait until all cells are executed.

The errors in our mathematical model are assumed to be realizations from a normal distribution. This implies that if we would be able to repeat the same set of measurements, every time the outcomes will be different. How different depends on the precision of the observations.

In this notebook we will demonstrate this by sampling different outcomes. Everytime you click for a new sample, you will see in the left figure that the observations are different, and therefore the fitted line as well. The truth that is used to generate the samples is also shown. On the right we will store each realization of the fitted model, and also show the average (red line) of those. You will see that it slowly converges to the truth (dashed black line).

In the bottom figure you can play with the slider and see that with many samples:
* the average of all fitted models nicely converged to the truth
* the combination of all fitted lines indeed looks like the confidence interval you've seen before

In [11]:
import numpy as np
import scipy as sc
from scipy.stats import norm
import matplotlib.pyplot as plt
%matplotlib inline
from ipywidgets import Button, VBox, Output

In [12]:

# BLUE function
def blue(A, y, Sigma_Y):
    W = np.linalg.inv(Sigma_Y)
    x_hat = np.linalg.inv(A.T @ W @ A) @ (A.T @ W @ y)
    return x_hat

# Set up observation model
np.random.seed(42)
m = 20
t = np.linspace(0, 10, m)
A = np.column_stack((np.ones(m), t))

# the "true" parameters and "true observations"
x_true = np.array([5.0, 0.4])
sigma = 1.5
Sigma_Y = sigma**2 * np.eye(m)
y_true = A @ x_true

# Generate a single realisation (with noise)
y = A @ x_true + np.random.normal(0, sigma, m)
#estimated parameters
x_hat = blue(A, y, Sigma_Y)
y_hat = A @ x_hat


# Storage for single realisations results 
x_estimates = []
y_estimates = []

# Plot function for a single realization 
def plot_single_realization(A, t, x_true, sigma, x_estimates):
    y = A @ x_true + np.random.normal(0, sigma, len(t))
    x_hat = blue(A, y, sigma**2 * np.eye(len(t)))
    x_estimates.append(x_hat)
    y_hat = A @ x_hat
    y_estimates.append(y_hat)
    
    # Compute current mean and covariance of estimates
    X_array = np.array(x_estimates)
    x_mean = np.mean(X_array, axis=0)
    
    y_mean_fit = A @ x_mean

    # --- PLOTS ---
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))

    # Left: new realization and fitted trend
    ax[0].scatter(t, y, label='Observations', color='tab:blue')
    ax[0].plot(t, y_hat, color='tab:red', label='Fitted model')
    ax[0].plot(t, A @ x_true, 'k--', label='Truth')
    ax[0].set_title(f'Realisation {len(x_estimates)}')
    ax[0].legend()
    ax[0].set_xlabel('t')
    ax[0].set_ylabel('y')
    ax[0].set_xlim(0, 10)  # Fixed x-axis limits
    ax[0].set_ylim(4, 10)  # Fixed y-axis limits
    
    # Right: mean of all fits so far + 95% conf. region
    ax[1].plot(t, A @ x_true, 'k--', lw=2, label='True trend')
    for i in range(len(y_estimates)):
        ax[1].plot(t, y_estimates[i], 'gray', alpha=0.3)
    ax[1].plot(t, y_mean_fit, 'r-', lw=2, label='Mean fitted model')
    ax[1].set_title(f'Mean model after {len(x_estimates)} realisations')
    ax[1].legend()
    ax[1].set_xlabel('t')
    ax[1].set_xlim(0, 10)  # Fixed x-axis limits
    ax[1].set_ylim(4, 10)  # Fixed y-axis limits
    
    plt.tight_layout()
    plt.show()

# Interactive control
out = Output()

def resample(_):
    with out:
        out.clear_output(wait=True)
        plot_single_realization(A, t, x_true, sigma, x_estimates)

# Create button
button = Button(
    description="Click: New sample",
    button_style='info',
    tooltip='Click to generate a new noisy sample and update the mean trend.'
)
button.on_click(resample)

# Display initial realization 
with out:
    plot_single_realization(A, t, x_true, sigma, x_estimates)

# Display button and output area 
display(VBox([button, out]))


In [ ]:
from ipywidgets import interact, IntSlider
# BLUE function
def blue(A, y, Sigma_Y):
    W = np.linalg.inv(Sigma_Y)
    x_hat = np.linalg.inv(A.T @ W @ A) @ (A.T @ W @ y)
    Sigma_Yhat = A @ np.linalg.inv(A.T @ W @ A) @ A.T
    return x_hat, Sigma_Yhat

def conf_interval(Sigma_Yhat, conf_level):
    """ 
    Function to calculate confidence interval of fitted model
    conf_level is the confidence level as percentage (e.g., 95)
    Output: 
    CI_yhat  confidence bound of yhat 
    k        CI_yhat[i] = k * sigma_yhat[i]
    """
    alpha = 1 - conf_level/100
    k = norm.ppf(1-0.5*alpha)
    CI_yhat = k * np.sqrt(np.diagonal(Sigma_Yhat))
    return CI_yhat, k

# Set up observation model
np.random.seed(42)
m = 20
t = np.linspace(0, 10, m)
A = np.column_stack((np.ones(m), t))

# The "true" parameters and "true" observations
x_true = np.array([5.0, 0.4])
sigma = 1.5
Sigma_Y = sigma**2 * np.eye(m)
y_true = A @ x_true


# Function to run N realisations and plot result
def monte_carlo_simulation(N=100):
    x_estimates = []
    y_estimates = []
    
    for _ in range(N):
        y = A @ x_true + np.random.normal(0, sigma, len(t))
        x_hat, S = blue(A, y, sigma**2 * np.eye(len(t)))
        x_estimates.append(x_hat)
        y_estimates.append(A @ x_hat)
    
    x_hat_CI, Sigma_Yhat = blue(A, y_true, sigma**2 * np.eye(len(t)))
    CI_yhat, k = conf_interval(Sigma_Yhat, 95)

    # PLOT
    plt.figure(figsize=(10, 6))  # Larger figure size for better visibility
    
    # Plot N random individual realizations
    for i in range(N):
        plt.plot(t, y_estimates[i], color='gray', alpha=0.3)
    plt.plot(t, A @ x_true, 'b--', lw=2, label='True trend')
    plt.plot(t, A @ x_true + CI_yhat, 'r',label=f'95% conf.int.')
    plt.plot(t, A @ x_true - CI_yhat, 'r')
    plt.title(f'{N} fitted trends')
    plt.legend()
    plt.xlabel('t')
    plt.ylabel('y')
    plt.xlim(0, 10)
    plt.ylim(4, 11)
    
    plt.tight_layout()
    plt.show()

# Interactive slider for N
interact(monte_carlo_simulation, N=IntSlider(value=10, min=1, max=2000, step=10, description='Realisations'));

interactive(children=(IntSlider(value=10, description='Realisations', max=2000, min=1, step=10), Output()), _d…